In [1]:
import pandas as pd
import shap
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

import matplotlib.pyplot as plt
from jupyter_dash import JupyterDash


from dash import Dash, html, dcc, Input, Output, State
import plotly.express as px

import dash
import dash_bootstrap_components as dbc


In [2]:
credit_card_data = pd.read_csv("C:/Users/jorge/OneDrive/Escritorio/Proyectos/archive/UCI_Credit_Card.csv")

In [3]:
credit_card_data.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [4]:
credit_card_data.describe()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.000000,30000.000000,30000.000000,3.000000e+04,30000.00000,30000.000000,30000.000000,30000.000000,30000.000000
mean,15000.500000,167484.322667,1.603733,1.853133,1.551867,35.485500,-0.016700,-0.133767,-0.166200,-0.220667,...,43262.948967,40311.400967,38871.760400,5663.580500,5.921163e+03,5225.68150,4826.076867,4799.387633,5215.502567,0.221200
std,8660.398374,129747.661567,0.489129,0.790349,0.521970,9.217904,1.123802,1.197186,1.196868,1.169139,...,64332.856134,60797.155770,59554.107537,16563.280354,2.304087e+04,17606.96147,15666.159744,15278.305679,17777.465775,0.415062
min,1.000000,10000.000000,1.000000,0.000000,0.000000,21.000000,-2.000000,-2.000000,-2.000000,-2.000000,...,-170000.000000,-81334.000000,-339603.000000,0.000000,0.000000e+00,0.00000,0.000000,0.000000,0.000000,0.000000
25%,7500.750000,50000.000000,1.000000,1.000000,1.000000,28.000000,-1.000000,-1.000000,-1.000000,-1.000000,...,2326.750000,1763.000000,1256.000000,1000.000000,8.330000e+02,390.00000,296.000000,252.500000,117.750000,0.000000
50%,15000.500000,140000.000000,2.000000,2.000000,2.000000,34.000000,0.000000,0.000000,0.000000,0.000000,...,19052.000000,18104.500000,17071.000000,2100.000000,2.009000e+03,1800.00000,1500.000000,1500.000000,1500.000000,0.000000
75%,22500.250000,240000.000000,2.000000,2.000000,2.000000,41.000000,0.000000,0.000000,0.000000,0.000000,...,54506.000000,50190.500000,49198.250000,5006.000000,5.000000e+03,4505.00000,4013.250000,4031.500000,4000.000000,0.000000
max,30000.000000,1000000.000000,2.000000,6.000000,3.000000,79.000000,8.000000,8.000000,8.000000,8.000000,...,891586.000000,927171.000000,961664.000000,873552.000000,1.684259e+06,896040.00000,621000.000000,426529.000000,528666.000000,1.000000


In [5]:
credit_card_data.drop_duplicates(inplace=True)

In [6]:
credit_card_data.columns

Index(['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0',
       'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2',
       'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1',
       'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6',
       'default.payment.next.month'],
      dtype='object')

In [52]:
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col([
            html.H3("Selecciona columnas para regresión logística"),
            dcc.Dropdown(
                id="column-selector",
                options=[{"label": col, "value": col} for col in credit_card_data.columns if col not in ["default.payment.next.month", "ID", "target"]],
                multi=True,
                value=[credit_card_data.columns[1]]  # valores iniciales
            ),      
            dcc.Slider(
                id='obs-slider',
                min=0,
                max=int(len(credit_card_data)*0.3)-1,
                step=1,
                value=0,
                marks={i: str(i) for i in range(0, int(len(credit_card_data)*0.3), 50)}  # marcas cada 100
            ),

            html.Div(id="model-output")
        ], width=6),
        dbc.Col([
            dcc.Graph(id="scatter-plot"),   # <-- este faltaba            
            dcc.Graph(id="waterfall-plot")  # nuevo gráfico

        ], width=8),

        html.Div(id='force-plot')

    ])
])

@app.callback(
    [Output("model-output", "children"),
     Output('force-plot', 'children'),
     Output("scatter-plot", "figure"),
     Output("waterfall-plot", "figure")],
    [Input("column-selector", "value"),
     Input("obs-slider", "value")]
)

def run_logistic(selected_columns, idx):
    
    X = credit_card_data[selected_columns]
    y = credit_card_data['default.payment.next.month']

    # Entrenar modelo
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    model = LogisticRegression(max_iter=5000)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Reporte
    report = classification_report(y_test, y_pred, output_dict=False, zero_division = 0)

    # Heatmap (solo métricas por clase, sin accuracy/macro/weighted)
    corr = X.corr()
    fig = px.imshow(
        corr,
        text_auto=True,
        aspect="auto",
        color_continuous_scale="RdBu_r",
        title="Heatmap del Classification Report"
    )

    # 4. Crear un explicador SHAP
    masker = shap.maskers.Independent(X_train)

    explainer = shap.LinearExplainer(model, masker=masker)
    
    # 5. Calcular valores SHAP para el conjunto de prueba
    shap_values = explainer.shap_values(X_test)

    force_html = shap.force_plot(
        explainer.expected_value,
        shap_values[idx,:],
        X_test.iloc[idx,:],
        matplotlib=False
    ).html()

    force_component = html.Iframe(
        srcDoc=force_html,
        style={"width": "200%", "height": "4000px", "border": "none"}
    )

    explainer = shap.Explainer(model, X_train) 
    shap_values = explainer.shap_values(X_test) 
    
    # Generar waterfall plot para la primera observación 
    # fig_waterfall = shap.plots.waterfall(shap_values[0], show=False)

    plt.figure()
    shap.plots.waterfall(shap_values[idx], show=False)
    fig_waterfall = tls.mpl_to_plotly(plt.gcf())  # convertir a plotly
    plt.close()

    # explainer = shap.Explainer(model, X_train)
    # shap_values = explainer(X_test)

    # plt.figure()
    # shap.plots.waterfall(shap_values[idx], show=False)
    # fig_waterfall = tls.mpl_to_plotly(plt.gcf())  # convierte Matplotlib a Plotly
    # plt.close()

    return html.Pre(report), force_component, fig, fig_waterfall

In [53]:
if __name__ == "__main__":
    app.run(debug=True, port=8051)

[2026-03-04 08:33:17,804] ERROR in app: Exception on /_dash-update-component [POST]
Traceback (most recent call last):
  File "C:\ProgramData\anaconda3\Lib\site-packages\flask\app.py", line 917, in full_dispatch_request
    rv = self.dispatch_request()
  File "C:\ProgramData\anaconda3\Lib\site-packages\flask\app.py", line 902, in dispatch_request
    return self.ensure_sync(self.view_functions[rule.endpoint])(**view_args)  # type: ignore[no-any-return]
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^
  File "C:\Users\jorge\AppData\Roaming\Python\Python313\site-packages\dash\_get_app.py", line 17, in wrap
    return ctx.run(func, self, *args, **kwargs)
           ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\jorge\AppData\Roaming\Python\Python313\site-packages\dash\dash.py", line 1600, in dispatch
    response_data = ctx.run(partial_func)
  File "C:\Users\jorge\AppData\Roaming\Python\Python313\site-packages\dash\_callback.py", line 720, in add_context

<!-- ideas --
-- Poner los datos atipicos, cuantos son y de que columna
-- varianza, correlación y mostrar quien tiene más datos, la desviación, el rango 
-- Poner en el dashboard una parte para hacer histograma y que ellos eligan la columna(s) -->